In [1]:
import sys
sys.path.append("/home/george/Desktop/workspace/dnn_processor/")  # just to enable `dataset`
sys.path.append("/home/george/Desktop/workspace/dnn_processor/dataset/")  # just to enable `dataset`

import tensorflow as tf
import dataset
import numpy as np
import keras
from datetime import datetime
from keras import layers

2025-04-13 17:31:57.899046: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-13 17:31:57.926678: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-04-13 17:31:57.926699: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-04-13 17:31:57.927451: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-13 17:31:57.932032: I tensorflow/core/platform/cpu_feature_guar

In [2]:
DEFAULT_BATCH_SIZE = 256
EPOCHS = 100
DEFAULT_INPUT_SIZE = 128

train_images_rgx = "../../dataset/train_set/*/*.jpeg"
test_images_rgx = "../../dataset/test_set/*/*.jpeg"

train_dataset, test_dataset, num_classes = dataset.create_datasets(
    train_images_rgx, test_images_rgx, DEFAULT_BATCH_SIZE, DEFAULT_INPUT_SIZE
)

2025-04-13 17:31:59.092302: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-04-13 17:31:59.116483: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-04-13 17:31:59.116618: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

In [3]:
EXP_ID = "2x_wider_baseline_model"


model = keras.models.load_model(
    f"../handmade/saved_models/{EXP_ID}/best_model_t/model_clean.h5",
    compile=False,
)
model.compile(
    keras.optimizers.legacy.Adam(), loss=tf.keras.losses.CategoricalCrossentropy(True), metrics=["accuracy"]
)

score = model.evaluate(test_dataset)
print("Accuracy: {}".format(score[1]))

model.summary()

1/1 [==============================] - 0s 360ms/step - loss: 0.1127 - accuracy: 1.0000
Accuracy: 1.0
Model: "model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 128, 128, 3)]     0         
                                                                 
 conv2d (Conv2D)             (None, 32, 32, 8)         392       
                                                                 
 activation (Activation)     (None, 32, 32, 8)         0         
                                                                 
 conv2d_1 (Conv2D)           (None, 15, 15, 32)        2336      
                                                                 
 activation_1 (Activation)   (None, 15, 15, 32)        0         
                                                                 
 conv2d_2 (Conv2D)           (None, 15, 15, 10)        330       
                        

2025-04-13 17:31:59.694711: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8904
2025-04-13 17:31:59.744175: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory


# quantize model

In [4]:
import qkeras
from qkeras import *
from keras.layers import (
    Conv2D,
    DepthwiseConv2D,
    Activation,
    Dense,
    Flatten,
    MaxPooling2D,
    GlobalAveragePooling2D,
)


BIT_WIDTH = 8
DEFAULT_ALPHA = 1


def create_qkeras_model_from_keras_model(keras_model):
    x = x_in = keras.Input(shape=keras_model.layers[0].output.shape[1:])

    x = QActivation(quantized_relu(BIT_WIDTH, 0))(x)

    for l in keras_model.layers:
        if isinstance(l, Conv2D):
            qconv2d = QConv2D(
                l.filters,
                l.kernel_size,
                l.strides,
                l.padding,
                kernel_quantizer=quantized_bits(BIT_WIDTH, 0, 1, alpha=DEFAULT_ALPHA),
                bias_quantizer=quantized_bits(BIT_WIDTH, 0, 1, alpha=DEFAULT_ALPHA),
            )
            x = qconv2d(x)
            qconv2d.set_weights(l.get_weights())

        elif isinstance(l, DepthwiseConv2D):
            qDepthwiseConv2D = QDepthwiseConv2D(
                l.kernel_size,
                l.strides,
                l.padding,
                depthwise_quantizer=quantized_bits(
                    BIT_WIDTH, 0, 1, alpha=DEFAULT_ALPHA
                ),
                bias_quantizer=quantized_bits(BIT_WIDTH, 0, 1, alpha=DEFAULT_ALPHA),
            )
            x = qDepthwiseConv2D(x)
            qDepthwiseConv2D.set_weights(l.get_weights())

        elif isinstance(l, Activation):
            activation_name = l.get_config()["activation"]
            print(activation_name)
            if activation_name == "tanh":
                x = QActivation(quantized_tanh(BIT_WIDTH, True))(x)
            elif activation_name == "relu":
                x = QActivation(quantized_relu(BIT_WIDTH, 0))(x)
            elif activation_name == "linear":
                x = QActivation(quantized_bits(BIT_WIDTH, 0, 1))(x)
            else:
                print(f"{l.name} was not quantized")

        elif isinstance(l, Flatten):
            # necessary because the layer before Flatten doesnt have avtivation
            x = QActivation(quantized_bits(BIT_WIDTH - 4, 0, 1))(x)
            x = Flatten()(x)

        elif isinstance(l, Dense):
            qdense = QDense(
                l.units,
                kernel_quantizer=quantized_bits(
                    BIT_WIDTH - 4, 0, 1, alpha=DEFAULT_ALPHA
                ),
                bias_quantizer=quantized_bits(BIT_WIDTH, 0, 1, alpha=DEFAULT_ALPHA),
            )
            # just for testing the latency
            x = qdense(x)
            qdense.set_weights(l.get_weights())

        elif isinstance(l, MaxPooling2D):
            x = MaxPooling2D()(x)

        elif isinstance(l, GlobalAveragePooling2D):
            globalAveragePooling2D = GlobalAveragePooling2D()
            x = globalAveragePooling2D(x)
            x = QActivation(quantized_bits(BIT_WIDTH, 0, 1, alpha=DEFAULT_ALPHA))(x)
        else:
            print(f"{l.name} layer was not quantized...")

    qkeras_model = keras.models.Model(x_in, x)

    return qkeras_model


LR = 0.0001
qkeras_model = create_qkeras_model_from_keras_model(model)
qkeras_model.compile(
    keras.optimizers.Adam(LR),
    loss=keras.losses.CategoricalCrossentropy(True),
    metrics=["accuracy"],
)
qkeras_model.summary()

input_2 layer was not quantized...


2025-04-13 17:32:00.312297: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory


relu
relu
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 128, 128, 3)]     0         
                                                                 
 q_activation (QActivation)  (None, 128, 128, 3)       0         
                                                                 
 q_conv2d (QConv2D)          (None, 32, 32, 8)         392       
                                                                 
 q_activation_1 (QActivatio  (None, 32, 32, 8)         0         
 n)                                                              
                                                                 
 q_conv2d_1 (QConv2D)        (None, 15, 15, 32)        2336      
                                                                 
 q_activation_2 (QActivatio  (None, 15, 15, 32)        0         
 n)                                                

/home/george/Desktop/workspace/dnn_processor/.venv/lib/python3.10/site-packages/keras/src/initializers/initializers.py:120: UserWarning: The initializer HeNormal is unseeded and being called multiple times, which will return identical values each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initializer instance more than once.
  warnings.warn(


In [5]:
from qkeras.autoqkeras.utils import print_qmodel_summary
print_qmodel_summary(qkeras_model)

q_activation         quantized_relu(8,0)
q_conv2d             f=8 quantized_bits(8,0,1,alpha=1) quantized_bits(8,0,1,alpha=1) 
q_activation_1       quantized_relu(8,0)
q_conv2d_1           f=32 quantized_bits(8,0,1,alpha=1) quantized_bits(8,0,1,alpha=1) 
q_activation_2       quantized_relu(8,0)
q_conv2d_2           f=10 quantized_bits(8,0,1,alpha=1) quantized_bits(8,0,1,alpha=1) 
q_activation_3       quantized_bits(4,0,1)
q_dense              u=5 quantized_bits(4,0,1,alpha=1) quantized_bits(8,0,1,alpha=1) 



In [6]:
from qkeras.estimate import print_qstats

print_qstats(qkeras_model)
qkeras_model.evaluate(test_dataset)
# free memory
del model

Instructions for updating:
Use ref() instead.


/home/george/Desktop/workspace/dnn_processor/.venv/lib/python3.10/site-packages/keras/src/constraints.py:365: UserWarning: The `keras.constraints.serialize()` API should only be used for objects of type `keras.constraints.Constraint`. Found an instance of type <class 'qkeras.quantizers.quantized_bits'>, which may lead to improper serialization.
  warnings.warn(
/home/george/Desktop/workspace/dnn_processor/.venv/lib/python3.10/site-packages/keras/src/initializers/initializers.py:120: UserWarning: The initializer HeNormal is unseeded and being called multiple times, which will return identical values each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initializer instance more than once.
  warnings.warn(
Instructions for updating:
Use ref() instead.



Number of operations in model:
    q_conv2d                      : 393216 (smult_8_8)
    q_conv2d_1                    : 518400 (smult_8_8)
    q_conv2d_2                    : 72000 (smult_8_8)
    q_dense                       : 11250 (smult_4_4)

Number of operation types in model:
    smult_4_4                     : 11250
    smult_8_8                     : 983616

Weight profiling:
    q_conv2d_weights               : 384   (8-bit unit)
    q_conv2d_bias                  : 8     (8-bit unit)
    q_conv2d_1_weights             : 2304  (8-bit unit)
    q_conv2d_1_bias                : 32    (8-bit unit)
    q_conv2d_2_weights             : 320   (8-bit unit)
    q_conv2d_2_bias                : 10    (8-bit unit)
    q_dense_weights                : 11250 (4-bit unit)
    q_dense_bias                   : 5     (8-bit unit)

Weight sparsity:
... quantizing model
    q_conv2d                       : 0.0128
    q_conv2d_1                     : 0.1139
    q_conv2d_2                    

In [7]:
qkeras_model.fit(train_dataset, batch_size=256, epochs=10, validation_data=test_dataset)
trial_id = "qatwqkeras"
qkeras_model.save(f"qatwqkeras/saved_qmodels/{trial_id}.h5")

Epoch 1/10


2025-04-13 17:32:03.800622: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape ingradient_tape/model/q_activation_2/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
2025-04-13 17:32:04.261195: I external/local_xla/xla/service/service.cc:168] XLA service 0x5f20cedfeb90 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-04-13 17:32:04.261209: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce RTX 3060, Compute Capability 8.6
2025-04-13 17:32:04.264906: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1744576324.321227   34893 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


2/2 [==============================] - 2s 41ms/step - loss: 0.2966 - accuracy: 0.8711 - val_loss: 0.3210 - val_accuracy: 0.8889
Epoch 2/10
2/2 [==============================] - 0s 41ms/step - loss: 0.2967 - accuracy: 0.8770 - val_loss: 0.3210 - val_accuracy: 0.8889
Epoch 3/10
2/2 [==============================] - 0s 41ms/step - loss: 0.2839 - accuracy: 0.8789 - val_loss: 0.3210 - val_accuracy: 0.8889
Epoch 4/10
2/2 [==============================] - 0s 48ms/step - loss: 0.2848 - accuracy: 0.8789 - val_loss: 0.3210 - val_accuracy: 0.8889
Epoch 5/10
2/2 [==============================] - 0s 41ms/step - loss: 0.3057 - accuracy: 0.8652 - val_loss: 0.3210 - val_accuracy: 0.8889
Epoch 6/10
2/2 [==============================] - 0s 42ms/step - loss: 0.2962 - accuracy: 0.8711 - val_loss: 0.3210 - val_accuracy: 0.8889
Epoch 7/10
2/2 [==============================] - 0s 43ms/step - loss: 0.3049 - accuracy: 0.8691 - val_loss: 0.3210 - val_accuracy: 0.8889
Epoch 8/10
2/2 [======================

/home/george/Desktop/workspace/dnn_processor/.venv/lib/python3.10/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [10]:
from qkeras.utils import _add_supported_quantized_objects

co = {}
_add_supported_quantized_objects(co)

loaded_qkeras_model = keras.models.load_model(f"qatwqkeras/saved_qmodels/{trial_id}.h5", co)
loaded_qkeras_model.evaluate(test_dataset)

# free memory
del loaded_qkeras_model

/home/george/Desktop/workspace/dnn_processor/.venv/lib/python3.10/site-packages/keras/src/initializers/initializers.py:120: UserWarning: The initializer HeNormal is unseeded and being called multiple times, which will return identical values each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initializer instance more than once.
  warnings.warn(


1/1 [==============================] - 0s 255ms/step - loss: 0.3210 - accuracy: 0.8889


# get metrics

In [11]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

x_test = []
y_test = []
for b in test_dataset.as_numpy_iterator():
    x_test.append(b[0])
    y_test.append(b[1])

x_test = np.concatenate(x_test, 0)
y_test = np.concatenate(y_test, 0)

y_test = np.argmax(y_test, axis=1)
y_qpred = qkeras_model.predict(x_test)
y_qpred = np.argmax(y_qpred, axis=1)

print("accuracy_score", accuracy_score(y_test, y_qpred))
print("precision_score", precision_score(y_test, y_qpred, average="macro"))
print("recall_score", recall_score(y_test, y_qpred, average="macro"))
print("f1_score", f1_score(y_test, y_qpred, average="macro"))


1/1 [==============================] - 0s 12ms/step
accuracy_score 0.8888888888888888
precision_score 0.9333333333333333
recall_score 0.9
f1_score 0.8933333333333333
